# scVI default - human_pbmc_sequencing_techs

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import inspect
import json
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/scvi_py310/lib/R"
)
os.environ["R_LIBS_SITE"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R/library"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib")

OPTIMIZER_PATH = Path(
    PROJECT_ROOT,
    "scripts",
    "optuna_multi_metrics",
    "optuna_scvi",
    "optimize_scvi_multi_metrics.py",
)
sys.path.insert(0, str(OPTIMIZER_PATH.parent))

import anndata as ad
import numpy as np
import pandas as pd
import torch
import optimize_scvi_multi_metrics as scvi_workflow

assert Path(scvi_workflow.__file__).resolve() == OPTIMIZER_PATH


In [2]:
DATASET_NAME = 'human_pbmc_sequencing_techs'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEYS = ('Method',)
BATCH_KEY = BATCH_KEYS[0]
LABEL_KEY = 'CellType'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
USE_HVG = True
N_TOP_HVG = 2000
MAX_EPOCHS = 400

EMBED_KEY = "X_scvi"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scvi_metric_batch"
METRIC_LABEL_KEY = "scvi_metric_label"

assert DATASET_NAME == 'human_pbmc_sequencing_techs'
assert BATCH_KEYS == ('Method',)
assert LABEL_KEY == 'CellType'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_keys": list(BATCH_KEYS),
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
        "max_epochs": MAX_EPOCHS,
    },
    name="configuration",
)


dataset                             human_pbmc_sequencing_techs
data_path     /ubc/cs/research/beaver/projects/Single-cell-i...
batch_keys                                             [Method]
label_key                                              CellType
metrics       [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
use_hvg                                                    True
n_top_hvg                                                  2000
max_epochs                                                  400
Name: configuration, dtype: object

In [3]:
scvi_workflow.DATA_PATH = DATA_PATH
scvi_workflow.DATASET_NAME = DATASET_NAME
scvi_workflow.BATCH_KEYS = list(BATCH_KEYS)
scvi_workflow.BATCH_KEY = BATCH_KEY
scvi_workflow.LABEL_KEY = LABEL_KEY
scvi_workflow.USE_HVG = USE_HVG
scvi_workflow.N_TOP_HVG = N_TOP_HVG
scvi_workflow.MAX_EPOCHS = MAX_EPOCHS
scvi_workflow.N_NEIGHBORS = N_NEIGHBORS
scvi_workflow.EMBED_KEY = EMBED_KEY
scvi_workflow.CLUSTER_KEY = CLUSTER_KEY
scvi_workflow.METRIC_BATCH_KEY = METRIC_BATCH_KEY
scvi_workflow.METRIC_LABEL_KEY = METRIC_LABEL_KEY

scvi_workflow.USE_BAMI = "BAMI" in SELECTED_METRICS
scvi_workflow.USE_CASW = "CASW" in SELECTED_METRICS
scvi_workflow.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
scvi_workflow.USE_TRAJ = "TRAJ" in SELECTED_METRICS
scvi_workflow.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
scvi_workflow.USE_BRAS = "BRAS" in SELECTED_METRICS
scvi_workflow.USE_KBET = "KBET" in SELECTED_METRICS
scvi_workflow.USE_CPCR = "CPCR" in SELECTED_METRICS

scvi_workflow.validate_configuration(SELECTED_METRICS)
scvi_workflow.validate_dataset_contents()

pd.Series(
    {
        "optimizer": str(OPTIMIZER_PATH),
        "scvi_version": scvi_workflow.scvi.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    name="preflight",
)


Dataset preflight: n_obs=30975 n_batches=9 n_labels=9 counts_layer=True
ISOLISI preflight: min_batch_coverage=6 isolated_labels=['Plasmacytoid dendritic cell']
CPCR preflight: eligible_labels=9


optimizer         /ubc/cs/research/beaver/projects/Single-cell-i...
scvi_version                                                  1.3.3
cuda_available                                                 True
Name: preflight, dtype: object

In [4]:
constructor_signature = inspect.signature(
    scvi_workflow.scvi.model.SCVI.__init__
)
training_signature = inspect.signature(
    scvi_workflow.scvi.train.TrainingPlan.__init__
)
assert scvi_workflow.scvi.__version__ == "1.3.3"
assert constructor_signature.parameters["n_latent"].default == 10
assert constructor_signature.parameters["n_hidden"].default == 128
assert constructor_signature.parameters["n_layers"].default == 1
assert constructor_signature.parameters["dropout_rate"].default == 0.1
assert training_signature.parameters["lr"].default == 0.001

SCVI_PARAMS = {'n_latent': 10, 'n_hidden': 128, 'n_layers': 1, 'dropout_rate': 0.1, 'lr': 0.001}
assert tuple(SCVI_PARAMS) == ('n_latent', 'n_hidden', 'n_layers', 'dropout_rate', 'lr')

pd.Series(
    {
        **SCVI_PARAMS,
        "gene_likelihood": "zinb",
        "dispersion": "gene",
        "latent_distribution": "normal",
        "early_stopping": True,
        "max_epochs": MAX_EPOCHS,
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
    },
    name="default_setting",
)

def run_scvi_setting(adata, seed):
    return scvi_workflow.run_scvi(
        adata,
        n_latent=SCVI_PARAMS["n_latent"],
        n_hidden=SCVI_PARAMS["n_hidden"],
        n_layers=SCVI_PARAMS["n_layers"],
        dropout_rate=SCVI_PARAMS["dropout_rate"],
        lr=SCVI_PARAMS["lr"],
        seed=int(seed),
    )


In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    result = scvi_workflow.compute_joint_metrics(adata_raw, adata_int)
    if tuple(result) != (
        *SELECTED_METRICS,
        "bio_score",
        "batch_score",
        "joint_score",
    ):
        raise ValueError(f"Unexpected metric result keys: {tuple(result)}")
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scvi_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


[rank: 0] Seed set to 0


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.7123572308537167


optimised clustering against scvi_metric_label
optimal cluster resolution: 0.4
optimal score: 0.7655828790700856
BRAS...


seed           9.000000
BAMI           0.765583
CASW           0.579342
SCGRAPHM       0.900594
ISOLISI        0.975176
BRAS           0.755230
KBET           0.118661
CPCR           0.360586
bio_score      0.805174
batch_score    0.411492
joint_score    0.647701
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.781207,0.574984,0.873008,0.968028,0.753438,0.113551,0.445044,0.799307,0.437344,0.654522
1,1,0.767006,0.576855,0.880697,0.969507,0.751006,0.112133,0.403243,0.798517,0.422128,0.647961
2,2,0.762507,0.575070,0.904071,0.953883,0.756146,0.113080,0.402762,0.798883,0.423996,0.648928
3,3,0.738639,0.565927,0.862498,0.937623,0.764631,0.125276,0.483956,0.776172,0.457954,0.648885
4,4,0.762769,0.577355,0.897566,0.936081,0.749236,0.108944,0.414499,0.793443,0.424226,0.645756
5,5,0.747096,0.575356,0.903751,0.961457,0.745411,0.103463,0.380847,0.796915,0.409907,0.642112
6,6,0.777362,0.572308,0.888817,0.948455,0.760033,0.119447,0.463355,0.796735,0.447612,0.657086
7,7,0.769339,0.574015,0.883213,0.977557,0.776006,0.124464,0.466694,0.801031,0.455721,0.662907
8,8,0.749525,0.575918,0.887784,0.940345,0.757869,0.121827,0.433311,0.788393,0.437669,0.648103
9,9,0.765583,0.579342,0.900594,0.975176,0.755230,0.118661,0.360586,0.805174,0.411492,0.647701


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.762103,0.012731,0.738639,0.781207
CASW,0.574713,0.003443,0.565927,0.579342
SCGRAPHM,0.888200,0.013070,0.862498,0.904071
ISOLISI,0.956811,0.014912,0.936081,0.977557
BRAS,0.756901,0.008224,0.745411,0.776006
KBET,0.116085,0.006686,0.103463,0.125276
CPCR,0.425430,0.037856,0.360586,0.483956
bio_score,0.795457,0.007688,0.776172,0.805174
batch_score,0.432805,0.016325,0.409907,0.457954
joint_score,0.650396,0.005756,0.642112,0.662907
